# Asynchronous Processing with Redis

Not all inference workloads are interactive. Tasks like document
summarization, batch classification, and offline evaluation can tolerate
higher latency in exchange for better throughput and resource efficiency.

The **llm-d Async Processor** provides a queue-based architecture for
these latency-insensitive workloads:

1. Clients submit requests to a **Redis queue** instead of the live API.
2. The Async Processor pulls requests from Redis and sends them through
   the llm-d inference stack.
3. **Flow-control gating** ensures batch work does not impact interactive
   traffic. The processor backs off when the interactive queue is busy.
4. Results are written back to Redis for clients to retrieve.

This notebook covers deploying Redis, configuring the Async Processor,
submitting batch requests, and monitoring flow control behavior.

In [ ]:
# Environment setup
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["REDIS_NS"] = "llm-d"

print("Namespace:", os.environ["NAMESPACE"])
print("Redis namespace:", os.environ["REDIS_NS"])

## Architecture Overview

```
Interactive Traffic              Batch Traffic
       |                              |
       v                              v
+-------------+              +-----------------+
| Envoy       |              | Redis Queue     |
| Gateway     |              | (pending jobs)  |
+------+------+              +--------+--------+
       |                              |
       v                              v
+-------------+              +-----------------+
| EPP Router  |              | Async Processor |
+------+------+              | (flow-control   |
       |                     |  gated)         |
       |                     +--------+--------+
       |                              |
       +----------+-------------------+
                  |
                  v
          +---------------+
          | vLLM Replicas |
          +---------------+
```

The key insight is that the Async Processor and the interactive router
share the same vLLM backend. Flow control ensures interactive requests
always have priority, and the processor only sends batch requests when
there is spare capacity.

In [ ]:
# Step 1: Deploy Redis as the message queue

redis_manifest = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: redis
  namespace: llm-d
spec:
  replicas: 1
  selector:
    matchLabels:
      app: redis
  template:
    metadata:
      labels:
        app: redis
    spec:
      containers:
        - name: redis
          image: redis:7-alpine
          ports:
            - containerPort: 6379
          args:
            - redis-server
            - --maxmemory
            - 2gb
            - --maxmemory-policy
            - noeviction
          resources:
            requests:
              memory: 2Gi
              cpu: 500m
            limits:
              memory: 3Gi
---
apiVersion: v1
kind: Service
metadata:
  name: redis
  namespace: llm-d
spec:
  selector:
    app: redis
  ports:
    - port: 6379
      targetPort: 6379
"""

with open("/tmp/redis.yaml", "w") as f:
    f.write(redis_manifest)

!kubectl apply -f /tmp/redis.yaml

print("Redis deployed.")
!kubectl wait --for=condition=ready pod -l app=redis -n $NAMESPACE --timeout=120s
print("Redis is ready.")
print()
print("Redis configuration:")
print("  maxmemory: 2 GB (sufficient for thousands of queued requests)")
print("  maxmemory-policy: noeviction (reject new writes when full, do not drop jobs)")

In [ ]:
# Step 2: Deploy the llm-d Async Processor
# The processor pulls requests from Redis, sends them to the inference
# stack, and writes results back to Redis

processor_manifest = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: async-processor
  namespace: llm-d
spec:
  replicas: 1
  selector:
    matchLabels:
      app: async-processor
  template:
    metadata:
      labels:
        app: async-processor
    spec:
      containers:
        - name: processor
          image: ghcr.io/llm-d/async-processor:latest
          env:
            - name: REDIS_URL
              value: redis://redis.llm-d.svc:6379
            - name: INFERENCE_URL
              value: http://envoy-gateway.llm-d.svc:8080
            - name: INPUT_QUEUE
              value: llm-d:batch:pending
            - name: OUTPUT_QUEUE
              value: llm-d:batch:completed
            - name: CONCURRENCY
              value: "8"
            - name: FLOW_CONTROL_ENABLED
              value: "true"
            - name: FLOW_CONTROL_QUEUE_THRESHOLD
              value: "10"
            - name: FLOW_CONTROL_BACKOFF_MS
              value: "1000"
          resources:
            requests:
              memory: 512Mi
              cpu: 500m
"""

with open("/tmp/async-processor.yaml", "w") as f:
    f.write(processor_manifest)

!kubectl apply -f /tmp/async-processor.yaml

print("Async Processor deployed.")
!kubectl wait --for=condition=ready pod -l app=async-processor -n $NAMESPACE --timeout=120s
print("Async Processor is ready.")
print()
print("Configuration:")
print("  CONCURRENCY:                 8 (max parallel inference requests)")
print("  FLOW_CONTROL_ENABLED:        true")
print("  FLOW_CONTROL_QUEUE_THRESHOLD: 10 (back off when queue depth > 10)")
print("  FLOW_CONTROL_BACKOFF_MS:     1000 (wait 1s before retrying)")

## How Flow Control Works

The Async Processor monitors the EPP's queue depth metric. When the
interactive queue exceeds the threshold (10 requests), the processor
pauses sending new batch requests and waits for the queue to drain.

```
Queue depth < 10:  Processor sends batch requests at full concurrency
Queue depth >= 10: Processor pauses, waits 1 second, checks again
Queue depth < 10:  Processor resumes sending batch requests
```

This simple mechanism ensures interactive users never experience latency
spikes due to batch processing. The batch work fills in the gaps when
the GPU would otherwise be idle.

In [ ]:
# Step 3: Submit batch requests to Redis
import subprocess
import json
import time

# Define a batch of summarization tasks
batch_requests = [
    {
        "id": f"batch-{i:04d}",
        "model": "Qwen/Qwen3-32B",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 150,
    }
    for i, prompt in enumerate([
        "Summarize the key principles of object-oriented programming.",
        "Explain the difference between TCP and UDP.",
        "What are the SOLID principles in software engineering?",
        "Describe the CAP theorem in distributed systems.",
        "What is the difference between concurrency and parallelism?",
        "Explain how garbage collection works in Java.",
        "What is a hash table and how does it work?",
        "Describe the differences between SQL and NoSQL databases.",
        "What is containerization and how does Docker work?",
        "Explain the publish-subscribe messaging pattern.",
        "What are microservices and what problems do they solve?",
        "Describe the MapReduce programming model.",
        "What is eventual consistency and when is it acceptable?",
        "Explain how TLS encryption works.",
        "What is a load balancer and what algorithms can it use?",
        "Describe the actor model for concurrent computation.",
        "What is a bloom filter and when would you use one?",
        "Explain the concept of idempotency in APIs.",
        "What is sharding and how does it help with database scaling?",
        "Describe the circuit breaker pattern for microservices.",
    ])
]

# Push requests to the Redis pending queue
print(f"Submitting {len(batch_requests)} batch requests to Redis...")

for req in batch_requests:
    payload = json.dumps(req)
    subprocess.run(
        ["kubectl", "exec", "-n", "llm-d", "deploy/redis", "--",
         "redis-cli", "RPUSH", "llm-d:batch:pending", payload],
        capture_output=True
    )

print(f"  Submitted {len(batch_requests)} requests.")

# Check queue length
result = subprocess.run(
    ["kubectl", "exec", "-n", "llm-d", "deploy/redis", "--",
     "redis-cli", "LLEN", "llm-d:batch:pending"],
    capture_output=True, text=True
)
print(f"  Pending queue length: {result.stdout.strip()}")

In [ ]:
# Step 4: Monitor processing progress
import time

print("Monitoring batch processing progress...")
print()

for i in range(12):
    # Check pending queue
    pending = subprocess.run(
        ["kubectl", "exec", "-n", "llm-d", "deploy/redis", "--",
         "redis-cli", "LLEN", "llm-d:batch:pending"],
        capture_output=True, text=True
    ).stdout.strip()

    # Check completed queue
    completed = subprocess.run(
        ["kubectl", "exec", "-n", "llm-d", "deploy/redis", "--",
         "redis-cli", "LLEN", "llm-d:batch:completed"],
        capture_output=True, text=True
    ).stdout.strip()

    print(f"  [{(i+1)*10:>3}s] Pending: {pending:>3}, Completed: {completed:>3}")

    if int(pending) == 0:
        print("\n  All requests processed!")
        break

    time.sleep(10)

# Show a sample completed result
print()
print("=== Sample Completed Result ===")
result = subprocess.run(
    ["kubectl", "exec", "-n", "llm-d", "deploy/redis", "--",
     "redis-cli", "LINDEX", "llm-d:batch:completed", "0"],
    capture_output=True, text=True
)

if result.stdout.strip():
    completed_job = json.loads(result.stdout.strip())
    print(f"  ID: {completed_job.get('id', 'N/A')}")
    if 'response' in completed_job:
        resp = completed_job['response']
        if 'choices' in resp:
            content = resp['choices'][0]['message']['content']
            print(f"  Response: {content[:200]}...")
    print(f"  Status: {completed_job.get('status', 'N/A')}")
else:
    print("  (no completed results yet)")

In [ ]:
# Step 5: Demonstrate flow control in action
# Generate interactive traffic while batch processing is running
# to show how the processor backs off

import threading

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

# Submit more batch requests
print("Submitting 50 more batch requests...")
for i in range(50):
    payload = json.dumps({
        "id": f"batch-flow-{i:04d}",
        "model": "Qwen/Qwen3-32B",
        "messages": [{"role": "user", "content": f"Write a paragraph about topic {i}."}],
        "max_tokens": 100,
    })
    subprocess.run(
        ["kubectl", "exec", "-n", "llm-d", "deploy/redis", "--",
         "redis-cli", "RPUSH", "llm-d:batch:pending", payload],
        capture_output=True
    )

print("Batch requests submitted.")
print()

# Simultaneously generate interactive traffic
interactive_stats = {"count": 0, "ttfts": []}
stop_event = threading.Event()

def interactive_traffic():
    while not stop_event.is_set():
        start = time.time()
        subprocess.run(
            ["curl", "-s", "-m", "30",
             f"http://{GATEWAY_IP}:8080/v1/chat/completions",
             "-H", "Content-Type: application/json",
             "-d", json.dumps({
                 "model": "Qwen/Qwen3-32B",
                 "messages": [{"role": "user", "content": "What is 2+2?"}],
                 "max_tokens": 10,
             })],
            capture_output=True
        )
        ttft = time.time() - start
        interactive_stats["count"] += 1
        interactive_stats["ttfts"].append(ttft)

# Start 10 interactive traffic threads
print("Generating concurrent interactive traffic (10 threads)...")
threads = [threading.Thread(target=interactive_traffic, daemon=True)
           for _ in range(10)]
for t in threads:
    t.start()

# Monitor for 60 seconds
for i in range(6):
    time.sleep(10)
    pending = subprocess.run(
        ["kubectl", "exec", "-n", "llm-d", "deploy/redis", "--",
         "redis-cli", "LLEN", "llm-d:batch:pending"],
        capture_output=True, text=True
    ).stdout.strip()

    recent_ttfts = interactive_stats["ttfts"][-10:]
    avg_ttft = sum(recent_ttfts) / len(recent_ttfts) * 1000 if recent_ttfts else 0

    print(f"  [{(i+1)*10:>2}s] Batch pending: {pending:>3}, "
          f"Interactive requests: {interactive_stats['count']:>4}, "
          f"Avg TTFT: {avg_ttft:.0f}ms")

stop_event.set()

print()
print("Flow control ensured that interactive TTFT stayed low even while")
print("batch processing was running. The processor backs off when it")
print("detects interactive queue depth above the threshold.")

In [ ]:
# Monitor processor throughput and flow control events

print("=== Async Processor Metrics ===")
print()

# Check processor logs for flow control events
print("Recent flow control events:")
!kubectl logs -n llm-d -l app=async-processor --tail=20 \
    | grep -i 'flow.control\|backoff\|throttl' | tail -5

print()

# Throughput metrics
print("Processor throughput:")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=rate(async_processor_requests_total{namespace="llm-d"}[5m])' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    rps = float(r['value'][1])
    print(f'  Batch throughput: {rps:.2f} req/s')
" 2>/dev/null || echo "  (metrics not yet available)"

print()
print("Flow control backoff count:")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=async_processor_backoff_total{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    count = r['value'][1]
    print(f'  Total backoff events: {count}')
" 2>/dev/null || echo "  (metrics not yet available)"

print()
print("A healthy async processing setup shows:")
print("  - Steady batch throughput during low interactive traffic")
print("  - Backoff events during interactive traffic spikes")
print("  - Interactive TTFT unaffected by batch processing")

## Summary

Asynchronous processing with Redis enables efficient batch inference
alongside interactive workloads on shared GPU infrastructure.

### Architecture

- **Redis**: Message queue for pending and completed batch requests
- **Async Processor**: Pulls from Redis, sends to llm-d, writes results back
- **Flow Control**: Monitors EPP queue depth and backs off during
  interactive traffic spikes

### Key Configuration

| Parameter | Default | Description |
|-----------|---------|-------------|
| CONCURRENCY | 8 | Max parallel inference requests from the processor |
| FLOW_CONTROL_QUEUE_THRESHOLD | 10 | Back off when queue depth exceeds this |
| FLOW_CONTROL_BACKOFF_MS | 1000 | Wait time before retrying after backoff |

### Use Cases

- Document summarization pipelines
- Batch classification and labeling
- Offline model evaluation
- Data enrichment workflows

### Next Steps

- **Batch Gateway**: For OpenAI-compatible batch processing with
  `/v1/batches` and `/v1/files` endpoints.
- **Flow Control**: Fine-tune priority bands to control how batch
  and interactive traffic share GPU capacity.